#data pipeline


In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import random
import numpy as np
from PIL import Image
import time
import matplotlib.pyplot as plt
import shutil

print("CUDA available:", torch.cuda.is_available())
print("Device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU only — go to Runtime > Change runtime type > GPU")

from google.colab import drive
drive.mount('/content/drive')
PROJECT_DIR = "/content/drive/MyDrive/denoising_project"
import os
os.makedirs(PROJECT_DIR, exist_ok=True)

In [ ]:
os.chdir("/content")

# --- Set12 from KAIR ---
if not os.path.exists("KAIR"):
    os.system("git clone -q https://github.com/cszn/KAIR.git")

src = "/content/KAIR/testsets/set12"
dst = f"{PROJECT_DIR}/testsets/Set12"
os.makedirs(f"{PROJECT_DIR}/testsets", exist_ok=True)
if os.path.exists(src) and not os.path.exists(dst):
    shutil.copytree(src, dst)
    print(f"Copied Set12 ({len(os.listdir(dst))} images) -> {dst}")
elif os.path.exists(dst):
    print(f"Set12 already present ({len(os.listdir(dst))} images)")
else:
    print("WARNING: set12 not found in KAIR — run `find KAIR -iname '*set12*'` to check")

# --- BSD68 / CBSD68 from FFDNet ---
if not os.path.exists("FFDNet"):
    os.system("git clone -q --depth 1 https://github.com/cszn/FFDNet.git")

for name in ["BSD68", "CBSD68"]:
    src = f"/content/FFDNet/testsets/{name}"
    dst = f"{PROJECT_DIR}/testsets/{name}"
    if os.path.exists(src) and not os.path.exists(dst):
        shutil.copytree(src, dst)
        print(f"Copied {name} ({len(os.listdir(dst))} images) -> {dst}")
    elif os.path.exists(dst):
        print(f"{name} already present ({len(os.listdir(dst))} images)")
    else:
        print(f"WARNING: {name} not found in FFDNet repo — run `find FFDNet -iname '*{name}*'` to check")



In [ ]:
kodak_dir = f"{PROJECT_DIR}/testsets/Kodak24"
os.makedirs(kodak_dir, exist_ok=True)

for i in range(1, 25):
    fname = f"kodim{i:02d}.png"
    fpath = os.path.join(kodak_dir, fname)
    if not os.path.exists(fpath):
        os.system(f"wget -q -O {fpath} http://r0k.us/graphics/kodak/kodak/{fname}")

n_kodak = len([f for f in os.listdir(kodak_dir) if f.endswith(".png")])
print(f"Kodak24: {n_kodak}/24 images downloaded")

In [ ]:
import shutil

train_dir = f"{PROJECT_DIR}/trainsets/BSD400"
os.makedirs(train_dir, exist_ok=True)

if len([f for f in os.listdir(train_dir) if f.lower().endswith((".jpg", ".png"))]) == 0:
    os.chdir("/content")
    if not os.path.exists("BSR"):
        os.system("git clone -q https://github.com/BIDS/BSDS500.git BSR_mirror")

    found_any = False
    for root, dirs, files in os.walk("/content/BSR_mirror"):
        imgs = [f for f in files if f.lower().endswith((".jpg", ".png"))]
        if imgs and ("train" in root.lower() or "val" in root.lower()):
            for f in imgs:
                shutil.copy(os.path.join(root, f), train_dir)
            found_any = True
            print(f"Copied {len(imgs)} images from {root}")

    if not found_any:
        print("Mirror structure didn't match — listing repo contents for manual inspection:")
        os.system("find /content/BSR_mirror -type d")

n_train = len([f for f in os.listdir(train_dir) if f.lower().endswith((".jpg", ".png"))])
print(f"\nFinal count in {train_dir}: {n_train} training images")

In [ ]:
# --- Fix Data Leak: Remove all CBSD68 images from BSD400 training directory ---
cbsd_stems = {os.path.splitext(f)[0] for f in os.listdir(f"{PROJECT_DIR}/testsets/CBSD68")}
removed = 0
for f in os.listdir(train_dir):
    if os.path.splitext(f)[0] in cbsd_stems:
        os.remove(os.path.join(train_dir, f))
        removed += 1
print(f"{removed} leaked CBSD68 images removed; {len(os.listdir(train_dir))} training images remain")  # expect 68 and 232

In [ ]:
import numpy as np

def add_gaussian_noise(img: np.ndarray, sigma: float) -> np.ndarray:
    """img expected as float32 array in [0, 255]. sigma in same scale (e.g., 15/25/50)."""
    noise = np.random.normal(0, sigma, img.shape).astype(np.float32)
    return np.clip(img + noise, 0, 255)

def add_impulse_noise(img: np.ndarray, density: float) -> np.ndarray:
    """density = fraction of pixels corrupted, e.g. 0.05 to 0.30."""
    noisy = img.copy()
    mask = np.random.rand(*img.shape[:2]) < density
    salt_pepper = np.random.choice([0, 255], size=mask.sum())
    if noisy.ndim == 3:
        for c in range(noisy.shape[2]):
            channel = noisy[:, :, c]
            channel[mask] = salt_pepper
            noisy[:, :, c] = channel
    else:
        noisy[mask] = salt_pepper
    return noisy

def add_mixed_noise(img: np.ndarray, sigma: float, density: float) -> np.ndarray:
    """Gaussian first, then impulse on top — mimics compounded real-world corruption."""
    noisy = add_gaussian_noise(img, sigma)
    noisy = add_impulse_noise(noisy, density)
    return noisy

NOISE_TYPES = {
    0: "gaussian",
    1: "impulse",
    2: "mixed",
}

In [ ]:
from torch.utils.data import Dataset, DataLoader
from PIL import Image
import random

class DenoisingDataset(Dataset):
    def __init__(self, root_dir, patch_size=64, train=True,
                 sigma_range=(15, 50), density_range=(0.05, 0.30), repeats=4):
        self.paths = [os.path.join(root_dir, f) for f in os.listdir(root_dir)
                      if f.lower().endswith((".png", ".jpg", ".jpeg"))]
        self.patch_size = patch_size
        self.train = train
        self.sigma_range = sigma_range
        self.density_range = density_range
        self.repeats = repeats

    def __len__(self):
        return len(self.paths) * self.repeats

    def _random_crop(self, img: np.ndarray) -> np.ndarray:
        h, w = img.shape[:2]
        ps = self.patch_size
        if h < ps or w < ps:
            # pad small images up to patch size
            pad_h, pad_w = max(0, ps - h), max(0, ps - w)
            img = np.pad(img, ((0, pad_h), (0, pad_w), (0, 0)), mode="reflect")
            h, w = img.shape[:2]
        top = random.randint(0, h - ps)
        left = random.randint(0, w - ps)
        return img[top:top + ps, left:left + ps]

    def __getitem__(self, idx):
        img = Image.open(self.paths[idx % len(self.paths)]).convert("RGB")
        img = np.array(img).astype(np.float32)
        patch = self._random_crop(img)

        # randomly pick a noise type each time
        noise_type = random.choice([0, 1, 2])
        sigma = random.uniform(*self.sigma_range)
        density = random.uniform(*self.density_range)

        if noise_type == 0:
            noisy = add_gaussian_noise(patch, sigma)
            severity = sigma / self.sigma_range[1]  # normalized 0-1
            sigma_n, density_n = sigma / self.sigma_range[1], 0.0
        elif noise_type == 1:
            noisy = add_impulse_noise(patch, density)
            severity = density / self.density_range[1]
            sigma_n, density_n = 0.0, density / self.density_range[1]
        else:
            noisy = add_mixed_noise(patch, sigma * 0.6, density * 0.6)
            severity = 0.5 * (sigma * 0.6 / self.sigma_range[1] + density * 0.6 / self.density_range[1])
            sigma_n = sigma * 0.6 / self.sigma_range[1]
            density_n = density * 0.6 / self.density_range[1]

        # to CHW tensors, normalized to [0,1]
        clean_t = torch.from_numpy(patch / 255.0).permute(2, 0, 1).float()
        noisy_t = torch.from_numpy(noisy / 255.0).permute(2, 0, 1).float()
        # label = [noise_type, severity, gaussian_strength, impulse_density]
        label_t = torch.tensor([noise_type, severity, sigma_n, density_n], dtype=torch.float32)

        return noisy_t, clean_t, label_t

In [ ]:
train_dataset = DenoisingDataset(train_dir, patch_size=64, train=True)
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True,
                           num_workers=2, pin_memory=True)

print(f"Training set: {len(train_dataset)} images -> {len(train_loader)} batches/epoch")

noisy_batch, clean_batch, label_batch = next(iter(train_loader))
print("noisy batch:", noisy_batch.shape)
print("clean batch:", clean_batch.shape)
print("label batch (noise_type, severity):", label_batch.shape)
print("sample labels:\n", label_batch[:5])


In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(8, 4))
axes[0].imshow(clean_batch[0].permute(1, 2, 0).numpy())
axes[0].set_title("Clean patch")
axes[0].axis("off")

noise_names = {0: "gaussian", 1: "impulse", 2: "mixed"}
nt = int(label_batch[0, 0].item())
axes[1].imshow(noisy_batch[0].permute(1, 2, 0).numpy())
axes[1].set_title(f"Noisy patch ({noise_names[nt]})")
axes[1].axis("off")

plt.tight_layout()
plt.savefig(f"{PROJECT_DIR}/sanity_check.png", dpi=150)
plt.show()


In [ ]:
#Device + checkpoint folder setup
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Training on:", device)

CHECKPOINT_DIR = f"{PROJECT_DIR}/checkpoints"
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

In [ ]:
#PSNR metric (standard denoising evaluation metric)
def compute_psnr(pred: torch.Tensor, target: torch.Tensor) -> float:
    """pred, target expected in [0,1] range, shape [B,C,H,W]."""
    mse = torch.mean((pred - target) ** 2, dim=[1, 2, 3])
    mse = torch.clamp(mse, min=1e-10)  # avoid log(0)
    psnr = 10 * torch.log10(1.0 / mse)
    return psnr.mean().item()



In [ ]:
#Evaluate on Set12 / BSD68 / Kodak24 (held-out test sets)
from PIL import Image
import numpy as np

def load_test_images(folder):
    imgs = []
    for f in sorted(os.listdir(folder)):
        if f.lower().endswith((".png", ".jpg", ".jpeg")):
            img = Image.open(os.path.join(folder, f)).convert("RGB")
            imgs.append(np.array(img).astype(np.float32))
    return imgs

def evaluate_on_testset(model, folder, sigma=25, device=device):
    """Adds fixed-sigma Gaussian noise (standard evaluation protocol) and reports avg PSNR."""
    imgs = load_test_images(folder)
    model.eval()
    total_psnr = 0.0

    with torch.no_grad():
        for img in imgs:
            noisy_np = add_gaussian_noise(img, sigma)  # from Cell 5 in the data pipeline file
            clean_t = torch.from_numpy(img / 255.0).permute(2, 0, 1).unsqueeze(0).float().to(device)
            noisy_t = torch.from_numpy(noisy_np / 255.0).permute(2, 0, 1).unsqueeze(0).float().to(device)

            denoised_t = model(noisy_t).clamp(0, 1)
            total_psnr += compute_psnr(denoised_t, clean_t)

    return total_psnr / len(imgs)

#shared helpers (ECA, evaluation, training utilities)

In [ ]:
import pandas as pd

# ---- ONE unified ECA class used by every model in this notebook ----
class ECA(nn.Module):
    """Efficient Channel Attention (Wang et al., 2020).
    raw_logits() -> pre-sigmoid (B, C): lets the Noise-Adaptive model apply FiLM before the sigmoid
    forward()    -> sigmoid weights (B, C, 1, 1): used by all the baseline models"""
    def __init__(self, kernel_size=3):
        super().__init__()
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.conv = nn.Conv1d(1, 1, kernel_size, padding=(kernel_size - 1) // 2, bias=False)

    def raw_logits(self, x):
        y = self.pool(x).squeeze(-1).squeeze(-1)           # (B, C)
        return self.conv(y.unsqueeze(1)).squeeze(1)         # (B, C)

    def forward(self, x):
        return torch.sigmoid(self.raw_logits(x)).unsqueeze(-1).unsqueeze(-1)


# ---- evaluation helpers ----
TEST_FOLDER = f"{PROJECT_DIR}/testsets/CBSD68"

def evaluate_cross_domain(model, folder, noise_fn, noise_kwargs, device):
    imgs = load_test_images(folder)
    model.eval()
    total_psnr = 0.0
    with torch.no_grad():
        for img in imgs:
            noisy_np = noise_fn(img, **noise_kwargs)
            clean_t = torch.from_numpy(img / 255.0).permute(2, 0, 1).unsqueeze(0).float().to(device)
            noisy_t = torch.from_numpy(noisy_np / 255.0).permute(2, 0, 1).unsqueeze(0).float().to(device)
            denoised_t = model(noisy_t).clamp(0, 1)
            total_psnr += compute_psnr(denoised_t, clean_t)
    return total_psnr / len(imgs)

def evaluate_cross_domain_seeded(model, folder, noise_fn, noise_kwargs, device, seed):
    state = np.random.get_state()
    np.random.seed(seed)
    try:
        return evaluate_cross_domain(model, folder, noise_fn, noise_kwargs, device)
    finally:
        np.random.set_state(state)

def evaluate_repeated(model, folder, noise_fn, noise_kwargs, device, n_runs=5, base_seed=0):
    scores = np.array([evaluate_cross_domain_seeded(model, folder, noise_fn, noise_kwargs, device, base_seed + i)
                       for i in range(n_runs)])
    return scores.mean(), scores.std()

N_RUNS = 5

test_conditions = {
    "Gaussian": (add_gaussian_noise, {"sigma": 25}),
    "Impulse":  (add_impulse_noise, {"density": 0.15}),
    "Mixed":    (add_mixed_noise, {"sigma": 15, "density": 0.09}),
}


# ---- training helpers ----
class _DropLabelLoader:
    """DenoisingDataset yields (noisy, clean, label); train_model only needs (noisy, clean)."""
    def __init__(self, loader):
        self.loader = loader
    def __iter__(self):
        for noisy, clean, _label in self.loader:
            yield noisy, clean
    def __len__(self):
        return len(self.loader)

def train_model(model, loader, num_epochs, checkpoint_path, device, lr=1e-3):
    criterion = nn.MSELoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=30, gamma=0.5)
    best_psnr = 0.0
    for epoch in range(1, num_epochs + 1):
        model.train()
        epoch_loss, epoch_psnr = 0.0, 0.0
        start = time.time()
        for noisy, clean in loader:
            noisy, clean = noisy.to(device), clean.to(device)
            optimizer.zero_grad()
            denoised = model(noisy)
            loss = criterion(denoised, clean)
            loss.backward()
            optimizer.step()
            epoch_loss += loss.item()
            with torch.no_grad():
                epoch_psnr += compute_psnr(denoised.clamp(0, 1), clean)
        scheduler.step()
        avg_loss, avg_psnr = epoch_loss / len(loader), epoch_psnr / len(loader)
        print(f"  Epoch {epoch:03d}/{num_epochs} | Loss: {avg_loss:.5f} | PSNR: {avg_psnr:.2f} dB | Time: {time.time()-start:.1f}s")
        if avg_psnr > best_psnr:
            best_psnr = avg_psnr
            torch.save(model.state_dict(), checkpoint_path)
    model.load_state_dict(torch.load(checkpoint_path, map_location=device))
    print(f"  Best training PSNR: {best_psnr:.2f} dB, loaded best weights from {checkpoint_path}")
    return model

FAIR_EPOCHS = 50

#main architecture

In [ ]:
import torch.nn.functional as F

class HighPass(nn.Module):
    """Fixed (NOT learned) high-pass filter: x - blur(x).
    Removes most image content and keeps the noise, so the head judges the noise itself."""
    def __init__(self):
        super().__init__()
        k = torch.tensor([1., 4., 6., 4., 1.])
        k = k[:, None] * k[None, :]
        k = k / k.sum()
        self.register_buffer("w", k.expand(3, 1, 5, 5).clone())

    def forward(self, x):
        return x - F.conv2d(F.pad(x, (2, 2, 2, 2), mode="reflect"), self.w, groups=3)


class ResidualFiLM(nn.Module):
    """gamma = 1 + dgamma, beta = beta, with the generator zero-initialised, so the model
    STARTS as plain ECA attention and learns the conditioning as a small correction.
    `mask` (B,1) switches conditioning off per sample (used for conditioning dropout)."""
    def __init__(self, desc_dim, channels):
        super().__init__()
        self.fc = nn.Linear(desc_dim, channels * 2)
        nn.init.zeros_(self.fc.weight)
        nn.init.zeros_(self.fc.bias)

    def forward(self, desc, mask):
        dgamma, beta = self.fc(desc).chunk(2, dim=1)
        return 1 + mask * dgamma, mask * beta

In [ ]:
class NoiseConditionedRAB(nn.Module):
    """Depthwise-separable convs + ECA channel attention. The attention logits are modulated by
    FiLM computed from the 2-number noise descriptor, BEFORE the sigmoid."""
    def __init__(self, channels, desc_dim=2, eca_kernel=3):
        super().__init__()
        self.dw1 = nn.Conv2d(channels, channels, 3, padding=1, groups=channels, bias=False)
        self.pw1 = nn.Conv2d(channels, channels, 1, bias=False)
        self.dw2 = nn.Conv2d(channels, channels, 3, padding=1, groups=channels, bias=False)
        self.pw2 = nn.Conv2d(channels, channels, 1, bias=False)
        self.act = nn.ReLU(inplace=True)
        self.eca = ECA(kernel_size=eca_kernel)          # unified ECA from the shared-helpers cell
        self.film = ResidualFiLM(desc_dim, channels)

    def forward(self, x, desc, mask):
        residual = x
        out = self.act(self.pw1(self.dw1(x)))
        out = self.pw2(self.dw2(out))
        gamma, beta = self.film(desc, mask)
        attn = torch.sigmoid(self.eca.raw_logits(out) * gamma + beta).unsqueeze(-1).unsqueeze(-1)
        return out * attn + residual

In [ ]:
class NoiseDescriptorHead(nn.Module):
    """Predicts a CONTINUOUS 2-number descriptor d = (gaussian strength, impulse density), each in [0,1].
    - works on a fixed high-pass residual, not on raw pixels
    - mean + std pooling: statistics that do not depend on image size (64x64 patches in training,
      full images at test time)
    - no categorical noise-type classifier, so unseen noise lands at an in-between point of the
      (gaussian, impulse) plane instead of being forced into one of three clusters"""
    def __init__(self, hidden=32):
        super().__init__()
        self.hp = HighPass()
        self.conv = nn.Sequential(
            nn.Conv2d(3, 16, 3, padding=1), nn.ReLU(inplace=True),
            nn.Conv2d(16, hidden, 3, padding=1), nn.ReLU(inplace=True),
        )
        self.fc = nn.Sequential(nn.Linear(2 * hidden, 32), nn.ReLU(inplace=True), nn.Linear(32, 2))

    def forward(self, x):
        f = self.conv(self.hp(x))
        pooled = torch.cat([f.mean(dim=(2, 3)), f.std(dim=(2, 3))], dim=1)
        return torch.sigmoid(self.fc(pooled))


class NoiseAdaptiveDenoiser(nn.Module):
    """Same call signature and 3-tuple return as before, so the training and evaluation cells keep working:
       returns (denoised, None, descriptor)."""
    def __init__(self, in_channels=3, num_features=48, num_blocks=6, embed_dim=None, p_drop=0.25):
        super().__init__()                      # embed_dim is accepted only for compatibility and ignored
        self.stem = nn.Conv2d(in_channels, num_features, 3, padding=1)
        self.noise_head = NoiseDescriptorHead()
        self.blocks = nn.ModuleList([NoiseConditionedRAB(num_features, desc_dim=2) for _ in range(num_blocks)])
        self.tail = nn.Conv2d(num_features, in_channels, 3, padding=1)
        self.p_drop = p_drop        # conditioning dropout: fraction of TRAINING samples run without conditioning
        self.cond_on = True         # at test time set False to switch conditioning off (diagnostic)

    def forward(self, x):
        desc = self.noise_head(x)
        if self.training and self.p_drop > 0:
            mask = (torch.rand(x.shape[0], 1, device=x.device) > self.p_drop).float()
        else:
            mask = torch.full((x.shape[0], 1), 1.0 if self.cond_on else 0.0, device=x.device)
        feat = self.stem(x)
        for block in self.blocks:
            feat = block(feat, desc, mask)
        return x - self.tail(feat), None, desc


_model = NoiseAdaptiveDenoiser()
_dummy = torch.randn(4, 3, 64, 64)
_denoised, _none, _desc = _model(_dummy)
print("denoised:", _denoised.shape)
print("descriptor:", _desc.shape, "(gaussian strength, impulse density)")
print(f"Total parameters: {sum(p.numel() for p in _model.parameters()):,}")
del _model, _dummy, _denoised, _none, _desc

In [ ]:
def combined_loss(denoised, clean, type_logits, descriptor, labels, aux_weight=0.1):
    """labels = [noise_type, severity, gaussian_strength, impulse_density] from DenoisingDataset.
    `type_logits` is unused (kept so the training cell does not change)."""
    recon_loss = nn.functional.mse_loss(denoised, clean)
    desc_loss = nn.functional.mse_loss(descriptor, labels[:, 2:4].float())
    total = recon_loss + aux_weight * desc_loss
    return total, recon_loss.item(), 0.0, desc_loss.item()

#comparison

HKCNN-style: parallel 3×3 and 5×5 kernel branches (the "heterogeneous kernel" idea), fused, then ECA attention

LWIFCNN-style: three parallel dilated branches at different rates (the "width fusion" idea), concatenated, then ECA attention

NAF-style: gate-based nonlinearity instead of ReLU, built from NAFNet's actual public-code components

Dual-Attention: channel attention then spatial/pixel attention applied sequentially (CBAM pattern), representing the paper's explicit "CA+PA" design

Cross-Channel: a full bottleneck FC attention instead of ECA's narrow local window — a reasonable, honest simplification standing in for QMSANet's richer cross-channel (quater

In [ ]:
# ---- 1. HKCNN-style: heterogeneous (multi-kernel) branches ----
class HKCNNStyleRAB(nn.Module):
    def __init__(self, channels):
        super().__init__()
        half = channels // 2
        self.branch3 = nn.Conv2d(channels, half, 3, padding=1, bias=False)
        self.branch5 = nn.Conv2d(channels, half, 5, padding=2, bias=False)
        self.fuse = nn.Conv2d(half * 2, channels, 1, bias=False)
        self.act = nn.ReLU(inplace=True)
        self.eca = ECA()
    def forward(self, x):
        residual = x
        out = self.act(self.fuse(torch.cat([self.branch3(x), self.branch5(x)], dim=1)))
        out = out * self.eca(out)
        return out + residual


# ---- 2. LWIFCNN-style: multi-branch width-information fusion ----
class LWIFCNNStyleRAB(nn.Module):
    def __init__(self, channels):
        super().__init__()
        third = channels // 3
        rem = channels - third * 2
        self.b1 = nn.Conv2d(channels, third, 3, padding=1, dilation=1, bias=False)
        self.b2 = nn.Conv2d(channels, third, 3, padding=2, dilation=2, bias=False)
        self.b3 = nn.Conv2d(channels, rem, 3, padding=3, dilation=3, bias=False)
        self.fuse = nn.Conv2d(channels, channels, 1, bias=False)
        self.act = nn.ReLU(inplace=True)
        self.eca = ECA()
    def forward(self, x):
        residual = x
        out = torch.cat([self.b1(x), self.b2(x), self.b3(x)], dim=1)
        out = self.act(self.fuse(out))
        out = out * self.eca(out)
        return out + residual


# ---- 3. NAF-style (SimpleGate + Simplified Channel Attention) ----
class SimpleGate(nn.Module):
    def forward(self, x):
        x1, x2 = x.chunk(2, dim=1)
        return x1 * x2

class SCA(nn.Module):
    def __init__(self, channels):
        super().__init__()
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.conv = nn.Conv2d(channels, channels, 1)
    def forward(self, x):
        return x * self.conv(self.pool(x))

class NAFBlock(nn.Module):
    def __init__(self, channels, expand_ratio=2):
        super().__init__()
        hidden = channels * expand_ratio
        self.norm1 = nn.GroupNorm(1, channels)
        self.conv1 = nn.Conv2d(channels, hidden, 1)
        self.dwconv = nn.Conv2d(hidden, hidden, 3, padding=1, groups=hidden)
        self.gate = SimpleGate()
        self.sca = SCA(hidden // 2)
        self.conv2 = nn.Conv2d(hidden // 2, channels, 1)
        self.beta = nn.Parameter(torch.zeros(1, channels, 1, 1))
    def forward(self, x):
        residual = x
        out = self.norm1(x)
        out = self.conv1(out)
        out = self.dwconv(out)
        out = self.gate(out)
        out = self.sca(out)
        out = self.conv2(out)
        return residual + out * self.beta


# ---- 4. Dual channel+pixel attention (CBAM-style) ----
class SpatialAttention(nn.Module):
    def __init__(self, kernel_size=7):
        super().__init__()
        self.conv = nn.Conv2d(2, 1, kernel_size, padding=kernel_size // 2, bias=False)
    def forward(self, x):
        avg_out = x.mean(dim=1, keepdim=True)
        max_out, _ = x.max(dim=1, keepdim=True)
        y = torch.cat([avg_out, max_out], dim=1)
        return torch.sigmoid(self.conv(y))

class DualAttentionRAB(nn.Module):
    def __init__(self, channels):
        super().__init__()
        self.dw1 = nn.Conv2d(channels, channels, 3, padding=1, groups=channels, bias=False)
        self.pw1 = nn.Conv2d(channels, channels, 1, bias=False)
        self.act = nn.ReLU(inplace=True)
        self.ca = ECA()
        self.pa = SpatialAttention()
    def forward(self, x):
        residual = x
        out = self.act(self.pw1(self.dw1(x)))
        out = out * self.ca(out)
        out = out * self.pa(out)
        return out + residual


# ---- 5. Cross-channel correlation attention (QMSANet-inspired) ----
class CrossChannelAttention(nn.Module):
    def __init__(self, channels, reduction=4):
        super().__init__()
        self.pool = nn.AdaptiveAvgPool2d(1)
        hidden = max(channels // reduction, 4)
        self.fc1 = nn.Linear(channels, hidden)
        self.fc2 = nn.Linear(hidden, channels)
        self.act = nn.ReLU(inplace=True)
    def forward(self, x):
        b, c, _, _ = x.shape
        y = self.pool(x).view(b, c)
        y = self.act(self.fc1(y))
        y = torch.sigmoid(self.fc2(y)).view(b, c, 1, 1)
        return x * y

class CrossChannelRAB(nn.Module):
    def __init__(self, channels):
        super().__init__()
        self.dw1 = nn.Conv2d(channels, channels, 3, padding=1, groups=channels, bias=False)
        self.pw1 = nn.Conv2d(channels, channels, 1, bias=False)
        self.act = nn.ReLU(inplace=True)
        self.cca = CrossChannelAttention(channels)
    def forward(self, x):
        residual = x
        out = self.act(self.pw1(self.dw1(x)))
        out = self.cca(out)
        return out + residual


# ---- generic wrapper: builds a full denoiser from any RAB block type ----
class GenericDenoiser(nn.Module):
    def __init__(self, rab_cls, in_channels=3, num_features=48, num_blocks=6):
        super().__init__()
        self.stem = nn.Conv2d(in_channels, num_features, 3, padding=1)
        self.blocks = nn.ModuleList([rab_cls(num_features) for _ in range(num_blocks)])
        self.tail = nn.Conv2d(num_features, in_channels, 3, padding=1)
    def forward(self, x):
        feat = self.stem(x)
        for block in self.blocks:
            feat = block(feat)
        residual = self.tail(feat)
        return x - residual


# quick sanity check for all five
_configs = [
    ("HKCNN-style", HKCNNStyleRAB),
    ("LWIFCNN-style", LWIFCNNStyleRAB),
    ("NAF-style", NAFBlock),
    ("Dual-Attention (CBAM-style)", DualAttentionRAB),
    ("Cross-Channel (QMSANet-style)", CrossChannelRAB),
]
for _name, _rab in _configs:
    _m = GenericDenoiser(_rab)
    _x = torch.randn(2, 3, 64, 64)
    _y = _m(_x)
    print(f"{_name:32s} output {tuple(_y.shape)}  params {sum(p.numel() for p in _m.parameters()):,}")
del _m, _x, _y

# Clean Multi-Seed Experimental Suite (3 Seeds, Best-Checkpoint Rule, Zero Data Leakage)

All models—**Main Architecture (`NoiseAdaptiveDenoiser`)**, **Literature Baselines (including Matched 44K `HKCNN-style`)**, **Cross-Domain Specialists**, and **6 Ablation Variants**—are trained with **3 independent random seeds** (`TRAIN_SEEDS = [42, 123, 456]`) on the deduplicated 232-image `BSD400` split, using the **exact same best-train-PSNR checkpoint selection**.

In [ ]:
import torch.nn.functional as F
from scipy import stats
import seaborn as sns

TRAIN_SEEDS = [42, 123, 456]
CLEAN_CKPT_ROOT = os.path.join(CHECKPOINT_DIR, "clean_3seed_v4")
RESULTS_DIR = os.path.join(PROJECT_DIR, "ablation_ieee_results_v4")
FIGURES_DIR = os.path.join(RESULTS_DIR, "figures")
CSV_DIR = os.path.join(RESULTS_DIR, "csv")

for _d in [CLEAN_CKPT_ROOT, RESULTS_DIR, FIGURES_DIR, CSV_DIR]:
    os.makedirs(_d, exist_ok=True)

FORCE_RETRAIN_ALL = False  # Set True to overwrite clean_3seed_v4 checkpoints


def set_all_seeds(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def make_seeded_train_loader(dataset, seed: int, batch_size=32):
    g = torch.Generator()
    g.manual_seed(seed)
    return DataLoader(
        dataset, batch_size=batch_size, shuffle=True,
        num_workers=2, pin_memory=torch.cuda.is_available(), generator=g
    )

## 1. Multi-Seed Training: `NoiseAdaptiveDenoiser` & Literature Baselines (Including 44K Matched `HKCNN-style`)

In [ ]:
def train_noise_adaptive_for_seed(seed: int, num_epochs=50, aux_weight=0.02):
    ckpt_path = os.path.join(CLEAN_CKPT_ROOT, f"noise_adaptive_seed{seed}.pth")
    set_all_seeds(seed)
    model = NoiseAdaptiveDenoiser(num_features=48, num_blocks=6, embed_dim=32).to(device)

    if os.path.exists(ckpt_path) and not FORCE_RETRAIN_ALL:
        print(f"[LOAD] Noise-Adaptive (ours) | seed={seed} <- {ckpt_path}")
        model.load_state_dict(torch.load(ckpt_path, map_location=device))
        return model

    print(f"[TRAIN] Noise-Adaptive (ours) | seed={seed}")
    loader = make_seeded_train_loader(train_dataset, seed)
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
    scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=30, gamma=0.5)
    best_psnr = 0.0

    for epoch in range(1, num_epochs + 1):
        model.train()
        ep_psnr = 0.0
        for noisy, clean, labels in loader:
            noisy, clean, labels = noisy.to(device), clean.to(device), labels.to(device)
            optimizer.zero_grad()
            denoised, type_logits, desc = model(noisy)
            loss, _, _, _ = combined_loss(denoised, clean, type_logits, desc, labels, aux_weight=aux_weight)
            loss.backward()
            optimizer.step()
            with torch.no_grad():
                ep_psnr += compute_psnr(denoised.clamp(0, 1), clean)
        scheduler.step()
        avg_psnr = ep_psnr / len(loader)
        if avg_psnr > best_psnr:
            best_psnr = avg_psnr
            torch.save(model.state_dict(), ckpt_path)
        if epoch % 10 == 0 or epoch == num_epochs:
            print(f"  Epoch {epoch:03d}/{num_epochs} | Train PSNR: {avg_psnr:.2f} dB")

    model.load_state_dict(torch.load(ckpt_path, map_location=device))
    print(f"  -> Best Train PSNR: {best_psnr:.2f} dB")
    return model


# Literature configs: (name, rab_cls, num_features)
LIT_CONFIGS = [
    ("HKCNN-style",                    HKCNNStyleRAB,    48),
    ("HKCNN-style (44K, matched)",     HKCNNStyleRAB,    20),
    ("LWIFCNN-style",                  LWIFCNNStyleRAB,  48),
    ("NAF-style",                      NAFBlock,         48),
    ("Dual-Attention (CBAM-style)",    DualAttentionRAB, 48),
    ("Cross-Channel (QMSANet-style)",  CrossChannelRAB,  48),
]

# Verify matched parameter count
_m_ours = NoiseAdaptiveDenoiser(num_features=48, num_blocks=6)
_m_hk_m = GenericDenoiser(HKCNNStyleRAB, num_features=20, num_blocks=6)
print(f"Noise-Adaptive (ours) params:        {sum(p.numel() for p in _m_ours.parameters()):,}")
print(f"HKCNN-style (44K, matched) params:   {sum(p.numel() for p in _m_hk_m.parameters()):,}")
del _m_ours, _m_hk_m

# Train / load all 3 seeds for Ours + 6 Literature models
lit_models_by_seed = {s: {} for s in TRAIN_SEEDS}

for seed in TRAIN_SEEDS:
    lit_models_by_seed[seed]["Noise-Adaptive (ours)"] = train_noise_adaptive_for_seed(seed, num_epochs=FAIR_EPOCHS)

    for name, rab_cls, nf in LIT_CONFIGS:
        safe = (
            name.lower().replace(" ", "_").replace("(", "").replace(")", "")
            .replace(",", "").replace("/", "_")
        )
        ckpt_path = os.path.join(CLEAN_CKPT_ROOT, f"{safe}_seed{seed}.pth")
        set_all_seeds(seed)
        model = GenericDenoiser(rab_cls, num_features=nf, num_blocks=6).to(device)
        if os.path.exists(ckpt_path) and not FORCE_RETRAIN_ALL:
            print(f"[LOAD] {name:32s} | seed={seed} <- {ckpt_path}")
            model.load_state_dict(torch.load(ckpt_path, map_location=device))
        else:
            print(f"[TRAIN] {name:32s} | seed={seed}")
            loader = make_seeded_train_loader(train_dataset, seed)
            model = train_model(model, _DropLabelLoader(loader), FAIR_EPOCHS, ckpt_path, device)
            # Ensure best-train-PSNR checkpoint is loaded
            model.load_state_dict(torch.load(ckpt_path, map_location=device))
        lit_models_by_seed[seed][name] = model

## 2. Cross-Domain Evidence: Single-Noise Specialists vs. `Noise-Adaptive (ours)`

Trains `Gaussian-only` and `Impulse-only` `LWIFCNN-style` specialists across `TRAIN_SEEDS` to provide reproducible cross-domain evidence alongside `Mixed-trained (LWIFCNN-style)` and `Noise-Adaptive (ours)`.

In [ ]:
class SingleNoiseDataset(Dataset):
    def __init__(self, root_dir, noise_type, patch_size=64,
                 sigma_range=(15, 50), density_range=(0.05, 0.30), repeats=4):
        assert noise_type in ("gaussian", "impulse")
        self.paths = sorted([
            os.path.join(root_dir, f) for f in os.listdir(root_dir)
            if f.lower().endswith((".png", ".jpg", ".jpeg"))
        ])
        self.noise_type = noise_type
        self.patch_size = patch_size
        self.sigma_range = sigma_range
        self.density_range = density_range
        self.repeats = repeats

    def __len__(self):
        return len(self.paths) * self.repeats

    def _random_crop(self, img):
        h, w = img.shape[:2]
        ps = self.patch_size
        if h < ps or w < ps:
            pad_h, pad_w = max(0, ps - h), max(0, ps - w)
            img = np.pad(img, ((0, pad_h), (0, pad_w), (0, 0)), mode="reflect")
            h, w = img.shape[:2]
        top = random.randint(0, h - ps)
        left = random.randint(0, w - ps)
        return img[top:top + ps, left:left + ps]

    def __getitem__(self, idx):
        img = Image.open(self.paths[idx % len(self.paths)]).convert("RGB")
        patch = self._random_crop(np.array(img).astype(np.float32))
        if self.noise_type == "gaussian":
            noisy = add_gaussian_noise(patch, random.uniform(*self.sigma_range))
        else:
            noisy = add_impulse_noise(patch, random.uniform(*self.density_range))
        clean_t = torch.from_numpy(patch / 255.0).permute(2, 0, 1).float()
        noisy_t = torch.from_numpy(noisy / 255.0).permute(2, 0, 1).float()
        return noisy_t, clean_t


gauss_ds = SingleNoiseDataset(train_dir, "gaussian", patch_size=64)
imp_ds   = SingleNoiseDataset(train_dir, "impulse",  patch_size=64)

cross_domain_models_by_seed = {s: {} for s in TRAIN_SEEDS}
for seed in TRAIN_SEEDS:
    for label, ds in [("Gaussian-only (LWIFCNN)", gauss_ds), ("Impulse-only (LWIFCNN)", imp_ds)]:
        safe = label.lower().replace(" ", "_").replace("(", "").replace(")", "").replace("-", "_")
        ckpt_path = os.path.join(CLEAN_CKPT_ROOT, f"{safe}_seed{seed}.pth")
        set_all_seeds(seed)
        m = GenericDenoiser(LWIFCNNStyleRAB, num_features=48, num_blocks=6).to(device)
        if os.path.exists(ckpt_path) and not FORCE_RETRAIN_ALL:
            print(f"[LOAD] {label:26s} | seed={seed}")
            m.load_state_dict(torch.load(ckpt_path, map_location=device))
        else:
            print(f"[TRAIN] {label:26s} | seed={seed}")
            loader = make_seeded_train_loader(ds, seed)
            m = train_model(m, loader, FAIR_EPOCHS, ckpt_path, device)
            m.load_state_dict(torch.load(ckpt_path, map_location=device))
        cross_domain_models_by_seed[seed][label] = m

    cross_domain_models_by_seed[seed]["Mixed-trained (LWIFCNN)"] = lit_models_by_seed[seed]["LWIFCNN-style"]
    cross_domain_models_by_seed[seed]["Noise-Adaptive (ours)"]   = lit_models_by_seed[seed]["Noise-Adaptive (ours)"]

## 3. Architectural Ablation Study (3 Seeds, Best-Checkpoint Rule)

Trains the 6 ablated variants across `TRAIN_SEEDS` (reusing `Noise-Adaptive (ours)` as `Full Model (Ours)` without redundant retraining). `Standard-Conv RAB` is excluded as requested.

In [ ]:
class AblationNoiseDescriptorHead(nn.Module):
    def __init__(self, hidden=32, use_highpass=True, pool_mode="mean_std"):
        super().__init__()
        assert pool_mode in ("mean_std", "mean_only")
        self.pool_mode = pool_mode
        self.hp = HighPass() if use_highpass else nn.Identity()
        self.conv = nn.Sequential(
            nn.Conv2d(3, 16, 3, padding=1), nn.ReLU(inplace=True),
            nn.Conv2d(16, hidden, 3, padding=1), nn.ReLU(inplace=True),
        )
        in_fc = 2 * hidden if pool_mode == "mean_std" else hidden
        self.fc = nn.Sequential(
            nn.Linear(in_fc, 32), nn.ReLU(inplace=True),
            nn.Linear(32, 2)
        )

    def forward(self, x):
        f = self.conv(self.hp(x))
        if self.pool_mode == "mean_std":
            pooled = torch.cat([f.mean(dim=(2, 3)), f.std(dim=(2, 3))], dim=1)
        else:
            pooled = f.mean(dim=(2, 3))
        return torch.sigmoid(self.fc(pooled))


class AblationRAB(nn.Module):
    def __init__(self, channels, desc_dim=2, eca_kernel=3,
                 use_film=True, film_mode="pre_sigmoid"):
        super().__init__()
        assert film_mode in ("pre_sigmoid", "post_sigmoid")
        self.use_film = use_film
        self.film_mode = film_mode
        self.conv1 = nn.Sequential(
            nn.Conv2d(channels, channels, 3, padding=1, groups=channels, bias=False),
            nn.Conv2d(channels, channels, 1, bias=False),
        )
        self.conv2 = nn.Sequential(
            nn.Conv2d(channels, channels, 3, padding=1, groups=channels, bias=False),
            nn.Conv2d(channels, channels, 1, bias=False),
        )
        self.act = nn.ReLU(inplace=True)
        self.eca = ECA(kernel_size=eca_kernel)
        self.film = ResidualFiLM(desc_dim, channels) if use_film else None

    def forward(self, x, desc=None, mask=None):
        residual = x
        out = self.act(self.conv1(x))
        out = self.conv2(out)
        if self.use_film and desc is not None:
            gamma, beta = self.film(desc, mask)
            if self.film_mode == "pre_sigmoid":
                attn = torch.sigmoid(self.eca.raw_logits(out) * gamma + beta).unsqueeze(-1).unsqueeze(-1)
            else:
                base_attn = torch.sigmoid(self.eca.raw_logits(out))
                attn = torch.clamp(base_attn * gamma + beta, 0.0, 1.0).unsqueeze(-1).unsqueeze(-1)
        else:
            attn = self.eca(out)
        return out * attn + residual


class ConfigurableNoiseAdaptiveDenoiser(nn.Module):
    def __init__(self, in_channels=3, num_features=48, num_blocks=6,
                 use_highpass=True, pool_mode="mean_std",
                 use_film=True, film_mode="pre_sigmoid", p_drop=0.25):
        super().__init__()
        self.use_film = use_film
        self.p_drop = p_drop
        self.cond_on = True
        self.stem = nn.Conv2d(in_channels, num_features, 3, padding=1)
        self.noise_head = AblationNoiseDescriptorHead(
            use_highpass=use_highpass, pool_mode=pool_mode
        ) if use_film else None
        self.blocks = nn.ModuleList([
            AblationRAB(num_features, desc_dim=2, eca_kernel=3, use_film=use_film, film_mode=film_mode)
            for _ in range(num_blocks)
        ])
        self.tail = nn.Conv2d(num_features, in_channels, 3, padding=1)

    def forward(self, x):
        if self.use_film:
            desc = self.noise_head(x)
            if self.training and self.p_drop > 0:
                mask = (torch.rand(x.shape[0], 1, device=x.device) > self.p_drop).float()
            else:
                mask = torch.full((x.shape[0], 1), 1.0 if self.cond_on else 0.0, device=x.device)
        else:
            desc, mask = None, None
        feat = self.stem(x)
        for block in self.blocks:
            feat = block(feat, desc, mask)
        return x - self.tail(feat), None, desc


ABLATION_CONFIGS = {
    "Full Model (Ours)": dict(
        kwargs=dict(use_highpass=True, pool_mode="mean_std", use_film=True,
                    film_mode="pre_sigmoid", p_drop=0.25),
        aux_weight=0.02,
    ),
    "w/o High-Pass": dict(
        kwargs=dict(use_highpass=False, pool_mode="mean_std", use_film=True,
                    film_mode="pre_sigmoid", p_drop=0.25),
        aux_weight=0.02,
    ),
    "w/o FiLM (Static)": dict(
        kwargs=dict(use_highpass=True, pool_mode="mean_std", use_film=False,
                    film_mode="pre_sigmoid", p_drop=0.0),
        aux_weight=0.0,
    ),
    "w/o Cond. Dropout": dict(
        kwargs=dict(use_highpass=True, pool_mode="mean_std", use_film=True,
                    film_mode="pre_sigmoid", p_drop=0.0),
        aux_weight=0.02,
    ),
    "w/o Aux Supervision": dict(
        kwargs=dict(use_highpass=True, pool_mode="mean_std", use_film=True,
                    film_mode="pre_sigmoid", p_drop=0.25),
        aux_weight=0.0,
    ),
    "Mean-Pool Only": dict(
        kwargs=dict(use_highpass=True, pool_mode="mean_only", use_film=True,
                    film_mode="pre_sigmoid", p_drop=0.25),
        aux_weight=0.02,
    ),
    "Post-Sigmoid FiLM": dict(
        kwargs=dict(use_highpass=True, pool_mode="mean_std", use_film=True,
                    film_mode="post_sigmoid", p_drop=0.25),
        aux_weight=0.02,
    ),
}

ablation_models_by_seed = {s: {} for s in TRAIN_SEEDS}
for seed in TRAIN_SEEDS:
    for vname, cfg in ABLATION_CONFIGS.items():
        if vname == "Full Model (Ours)":
            ablation_models_by_seed[seed][vname] = lit_models_by_seed[seed]["Noise-Adaptive (ours)"]
            continue

        safe = (
            vname.lower().replace(" ", "_").replace("/", "_")
            .replace("(", "").replace(")", "").replace(".", "")
        )
        ckpt_path = os.path.join(CLEAN_CKPT_ROOT, f"abl_{safe}_seed{seed}.pth")
        set_all_seeds(seed)
        m = ConfigurableNoiseAdaptiveDenoiser(**cfg["kwargs"]).to(device)
        if os.path.exists(ckpt_path) and not FORCE_RETRAIN_ALL:
            print(f"[LOAD] {vname:22s} | seed={seed}")
            m.load_state_dict(torch.load(ckpt_path, map_location=device))
        else:
            print(f"[TRAIN] {vname:22s} | seed={seed}")
            loader = make_seeded_train_loader(train_dataset, seed)
            optimizer = torch.optim.Adam(m.parameters(), lr=1e-3)
            scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=30, gamma=0.5)
            aux_w = cfg["aux_weight"]
            best_psnr = 0.0
            for epoch in range(1, FAIR_EPOCHS + 1):
                m.train()
                ep_psnr = 0.0
                for noisy, clean, labels in loader:
                    noisy, clean, labels = noisy.to(device), clean.to(device), labels.to(device)
                    optimizer.zero_grad()
                    denoised, type_logits, desc = m(noisy)
                    if aux_w > 0.0 and desc is not None:
                        loss, _, _, _ = combined_loss(denoised, clean, type_logits, desc, labels, aux_weight=aux_w)
                    else:
                        loss = F.mse_loss(denoised, clean)
                    loss.backward()
                    optimizer.step()
                    with torch.no_grad():
                        ep_psnr += compute_psnr(denoised.clamp(0, 1), clean)
                scheduler.step()
                avg_psnr = ep_psnr / len(loader)
                if avg_psnr > best_psnr:
                    best_psnr = avg_psnr
                    torch.save(m.state_dict(), ckpt_path)
                if epoch % 10 == 0 or epoch == FAIR_EPOCHS:
                    print(f"  Epoch {epoch:03d}/{FAIR_EPOCHS} | Train PSNR: {avg_psnr:.2f} dB")
            m.load_state_dict(torch.load(ckpt_path, map_location=device))
        ablation_models_by_seed[seed][vname] = m

## 4. Full Evaluation: 3 Datasets, Multi-Severity, Cross-Domain & Out-of-Range / Unseen Noise

Evaluates all 3 training seeds on:
- **In-Range Benchmarks**: `CBSD68` (held-out), `Set12`, `Kodak24` across Gaussian, Impulse, Mixed (PSNR & SSIM)
- **9-Level Severity Sweep** on `CBSD68`
- **Cross-Domain Evaluation** (`Gaussian-only`, `Impulse-only`, `Mixed-trained`, `Noise-Adaptive`)
- **Out-of-Range & Unseen Noise**: Gaussian $\sigma \in \{10, 65\}$, Impulse $p=0.40$, Mixed $(\sigma=35, p=0.20)$, **Speckle**, **Poisson**, and **Gaussian+Speckle**
- **Complexity**: Params (K), GMac @ 256×256, Latency (ms)

In [ ]:
TEST_DATASETS = {
    "CBSD68": os.path.join(PROJECT_DIR, "testsets", "CBSD68"),
    "Set12": os.path.join(PROJECT_DIR, "testsets", "Set12"),
    "Kodak24": os.path.join(PROJECT_DIR, "testsets", "Kodak24"),
}

_SSIM_WIN_CACHE = {}


def _get_ssim_window(window_size: int = 11, sigma: float = 1.5, channels: int = 3, dev="cpu"):
    key = (window_size, sigma, channels, str(dev))
    if key not in _SSIM_WIN_CACHE:
        coords = torch.arange(window_size, dtype=torch.float32, device=dev) - window_size // 2
        g = torch.exp(-(coords ** 2) / (2 * sigma ** 2))
        g = g / g.sum()
        win = (g[:, None] * g[None, :]).unsqueeze(0).unsqueeze(0).expand(channels, 1, window_size, window_size).contiguous()
        _SSIM_WIN_CACHE[key] = win
    return _SSIM_WIN_CACHE[key]


def compute_psnr_and_ssim_batch(pred: torch.Tensor, target: torch.Tensor, window_size: int = 11):
    """Returns per-image PSNR (B,) and SSIM (B,) tensors on GPU."""
    mse = torch.mean((pred - target) ** 2, dim=[1, 2, 3]).clamp(min=1e-10)
    psnr_vec = 10.0 * torch.log10(1.0 / mse)

    c = pred.shape[1]
    win = _get_ssim_window(window_size, 1.5, c, pred.device)
    pad = window_size // 2
    mu1 = F.conv2d(pred, win, padding=pad, groups=c)
    mu2 = F.conv2d(target, win, padding=pad, groups=c)
    mu1_sq, mu2_sq, mu1_mu2 = mu1 ** 2, mu2 ** 2, mu1 * mu2
    sigma1_sq = F.conv2d(pred * pred, win, padding=pad, groups=c) - mu1_sq
    sigma2_sq = F.conv2d(target * target, win, padding=pad, groups=c) - mu2_sq
    sigma12 = F.conv2d(pred * target, win, padding=pad, groups=c) - mu1_mu2
    C1, C2 = 0.01 ** 2, 0.03 ** 2
    ssim_map = ((2 * mu1_mu2 + C1) * (2 * sigma12 + C2)) / ((mu1_sq + mu2_sq + C1) * (sigma1_sq + sigma2_sq + C2) + 1e-12)
    ssim_vec = ssim_map.mean(dim=[1, 2, 3])
    return psnr_vec, ssim_vec


def count_params_and_macs(model: nn.Module, input_size=(1, 3, 256, 256), dev="cpu"):
    total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    macs = [0]
    hooks = []

    def conv_hook(mod, inp, out):
        b, c_out, h_out, w_out = out.shape
        k_h, k_w = mod.kernel_size
        macs[0] += b * c_out * h_out * w_out * ((mod.in_channels // mod.groups) * k_h * k_w)

    def linear_hook(mod, inp, out):
        b = inp[0].shape[0] if inp[0].ndim > 1 else 1
        macs[0] += b * mod.in_features * mod.out_features

    for m in model.modules():
        if isinstance(m, nn.Conv2d):
            hooks.append(m.register_forward_hook(conv_hook))
        elif isinstance(m, nn.Linear):
            hooks.append(m.register_forward_hook(linear_hook))

    model.eval()
    with torch.no_grad():
        _ = model(torch.randn(*input_size, device=dev))
    for h in hooks:
        h.remove()
    return total_params, macs[0] / 1e9


def measure_latency_ms(model: nn.Module, input_size=(1, 3, 256, 256), dev="cpu", warmup=10, runs=30):
    model.eval()
    x = torch.randn(*input_size, device=dev)
    with torch.no_grad():
        for _ in range(warmup):
            _ = model(x)
        if str(dev).startswith("cuda"):
            torch.cuda.synchronize()
        t0 = time.perf_counter()
        for _ in range(runs):
            _ = model(x)
        if str(dev).startswith("cuda"):
            torch.cuda.synchronize()
        t1 = time.perf_counter()
    return (t1 - t0) / runs * 1000.0


# ---- Out-of-Range & Unseen Noise Functions ----
def add_speckle_noise(img: np.ndarray, var: float = 0.08) -> np.ndarray:
    noise = np.random.normal(0.0, np.sqrt(var), img.shape).astype(np.float32)
    return np.clip(img + img * noise, 0, 255)


def add_poisson_noise(img: np.ndarray, scale: float = 30.0) -> np.ndarray:
    img01 = np.clip(img / 255.0, 0.0, 1.0)
    noisy01 = np.random.poisson(img01 * scale).astype(np.float32) / scale
    return np.clip(noisy01 * 255.0, 0, 255)


def add_gaussian_speckle_compound(img: np.ndarray, sigma: float = 20.0, var: float = 0.04) -> np.ndarray:
    return add_speckle_noise(add_gaussian_noise(img, sigma), var=var)


SEVERITY_SWEEP = {
    "Gaussian_s15": (add_gaussian_noise, {"sigma": 15}, "Gaussian", "s=15"),
    "Gaussian_s25": (add_gaussian_noise, {"sigma": 25}, "Gaussian", "s=25"),
    "Gaussian_s50": (add_gaussian_noise, {"sigma": 50}, "Gaussian", "s=50"),
    "Impulse_p05":  (add_impulse_noise,  {"density": 0.05}, "Impulse", "p=0.05"),
    "Impulse_p15":  (add_impulse_noise,  {"density": 0.15}, "Impulse", "p=0.15"),
    "Impulse_p30":  (add_impulse_noise,  {"density": 0.30}, "Impulse", "p=0.30"),
    "Mixed_low":    (add_mixed_noise,    {"sigma": 10, "density": 0.06}, "Mixed", "s10+p0.06"),
    "Mixed_mid":    (add_mixed_noise,    {"sigma": 15, "density": 0.09}, "Mixed", "s15+p0.09"),
    "Mixed_high":   (add_mixed_noise,    {"sigma": 25, "density": 0.15}, "Mixed", "s25+p0.15"),
}

OOR_UNSEEN_CONDITIONS = {
    "OOR_Gauss_s10":    (add_gaussian_noise, {"sigma": 10}, "Out-of-Range"),
    "OOR_Gauss_s65":    (add_gaussian_noise, {"sigma": 65}, "Out-of-Range"),
    "OOR_Imp_p40":      (add_impulse_noise,  {"density": 0.40}, "Out-of-Range"),
    "OOR_Mixed_s35p20": (add_mixed_noise,    {"sigma": 35, "density": 0.20}, "Out-of-Range"),
    "Unseen_Speckle":   (add_speckle_noise,  {"var": 0.08}, "Unseen"),
    "Unseen_Poisson":   (add_poisson_noise,  {"scale": 30.0}, "Unseen"),
    "Unseen_GaussSpeck":(add_gaussian_speckle_compound, {"sigma": 20.0, "var": 0.04}, "Unseen"),
}


def build_batched_noisy_cache(imgs, noise_fn, noise_kwargs, test_seed=100):
    """Pre-generates noisy images once and groups by (H, W) into GPU batches for ~25x faster eval."""
    state = np.random.get_state()
    np.random.seed(test_seed)
    groups = {}
    try:
        for img in imgs:
            noisy_np = noise_fn(img, **noise_kwargs)
            hw = img.shape[:2]
            if hw not in groups:
                groups[hw] = ([], [])
            groups[hw][0].append(torch.from_numpy(img / 255.0).permute(2, 0, 1))
            groups[hw][1].append(torch.from_numpy(noisy_np / 255.0).permute(2, 0, 1))
    finally:
        np.random.set_state(state)

    batches = []
    for hw, (c_list, n_list) in groups.items():
        batches.append((torch.stack(c_list).float().to(device), torch.stack(n_list).float().to(device)))
    return batches


def eval_cached_batches(model, batches, chunk_size=16):
    """Evaluates model over pre-built GPU batches in chunks to avoid OOM."""
    model.eval()
    all_psnr, all_ssim = [], []
    with torch.no_grad():
        for clean_b, noisy_b in batches:
            for start in range(0, clean_b.shape[0], chunk_size):
                c_sub = clean_b[start:start + chunk_size]
                n_sub = noisy_b[start:start + chunk_size]
                out = model(n_sub)
                den_b = (out[0] if isinstance(out, tuple) else out).clamp(0, 1)
                p_vec, s_vec = compute_psnr_and_ssim_batch(den_b, c_sub)
                all_psnr.append(p_vec)
                all_ssim.append(s_vec)
    p_all = torch.cat(all_psnr)
    s_all = torch.cat(all_ssim)
    return float(p_all.mean().item()), float(s_all.mean().item())


loaded_datasets = {k: load_test_images(v) for k, v in TEST_DATASETS.items()}
cbsd_imgs = loaded_datasets["CBSD68"]

# 1. Complexity (computed once from seed 42)
comp_rows = []
for suite, mdict in [("Literature", lit_models_by_seed[TRAIN_SEEDS[0]]), ("Ablation", ablation_models_by_seed[TRAIN_SEEDS[0]])]:
    for mname, mobj in mdict.items():
        params, gmacs = count_params_and_macs(mobj, input_size=(1, 3, 256, 256), dev=device)
        lat = measure_latency_ms(mobj, input_size=(1, 3, 256, 256), dev=device)
        comp_rows.append({
            "Suite": suite, "Model": mname, "Params": params,
            "Params_K": round(params / 1e3, 2), "GMac_256": round(gmacs, 3), "Latency_ms": round(lat, 2),
        })
df_complexity = pd.DataFrame(comp_rows)

# 2. Multi-dataset x Primary conditions across ALL 3 training seeds (Batched & Cached)
multids_seed_rows = []
per_seed_cbsd_means = {}

for ds_name, imgs in loaded_datasets.items():
    t_ds = time.time()
    print(f"Evaluating 3 seeds on {ds_name} ({len(imgs)} images)...")
    # Pre-build GPU batches once per condition (2 fixed test draws)
    cond_batches = {
        cname: [build_batched_noisy_cache(imgs, nfn, nkw, test_seed=1000 + d_i) for d_i in range(2)]
        for cname, (nfn, nkw) in test_conditions.items()
    }
    for seed in TRAIN_SEEDS:
        entries = (
            [("Literature", k, v, True) for k, v in lit_models_by_seed[seed].items()] +
            [("Ablation",   k, v, True) for k, v in ablation_models_by_seed[seed].items() if k != "Full Model (Ours)"] +
            [("Ablation", "Full Model (cond_on=False)", ablation_models_by_seed[seed]["Full Model (Ours)"], False)]
        )
        for suite, mname, mobj, cond_flag in entries:
            if hasattr(mobj, "cond_on"):
                mobj.cond_on = cond_flag
            for cname, draw_list in cond_batches.items():
                p_list, s_list = [], []
                for batches in draw_list:
                    pm, sm = eval_cached_batches(mobj, batches)
                    p_list.append(pm); s_list.append(sm)
                pm, sm = float(np.mean(p_list)), float(np.mean(s_list))
                multids_seed_rows.append({
                    "Suite": suite, "Model": mname, "Train_Seed": seed,
                    "Dataset": ds_name, "Condition": cname, "PSNR": pm, "SSIM": sm,
                })
                if ds_name == "CBSD68":
                    per_seed_cbsd_means.setdefault((suite, mname, cname), []).append(pm)
                # Mirror Noise-Adaptive (ours) as Full Model (Ours) in Ablation
                if suite == "Literature" and mname == "Noise-Adaptive (ours)":
                    multids_seed_rows.append({
                        "Suite": "Ablation", "Model": "Full Model (Ours)", "Train_Seed": seed,
                        "Dataset": ds_name, "Condition": cname, "PSNR": pm, "SSIM": sm,
                    })
                    if ds_name == "CBSD68":
                        per_seed_cbsd_means.setdefault(("Ablation", "Full Model (Ours)", cname), []).append(pm)
            if hasattr(mobj, "cond_on"):
                mobj.cond_on = True
    del cond_batches
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    print(f"  -> {ds_name} done in {time.time() - t_ds:.1f}s")

df_multids_seeds = pd.DataFrame(multids_seed_rows)
df_multids = df_multids_seeds.groupby(["Suite", "Model", "Dataset", "Condition"], as_index=False).agg(
    PSNR_mean=("PSNR", "mean"), PSNR_std=("PSNR", "std"),
    SSIM_mean=("SSIM", "mean"), SSIM_std=("SSIM", "std"),
)
df_multids["Cell"] = df_multids["Dataset"] + "_" + df_multids["Condition"]

# 3. Cross-domain evaluation across 3 seeds on CBSD68 (Batched & Cached)
t_cd = time.time()
cd_batches = {cname: build_batched_noisy_cache(cbsd_imgs, nfn, nkw, test_seed=2026) for cname, (nfn, nkw) in test_conditions.items()}
cd_seed_rows = []
for seed in TRAIN_SEEDS:
    for mname, mobj in cross_domain_models_by_seed[seed].items():
        for cname, batches in cd_batches.items():
            pm, sm = eval_cached_batches(mobj, batches)
            cd_seed_rows.append({"Model": mname, "Train_Seed": seed, "Condition": cname, "PSNR": pm, "SSIM": sm})
del cd_batches
df_cross_domain = pd.DataFrame(cd_seed_rows).groupby(["Model", "Condition"], as_index=False).agg(
    PSNR_mean=("PSNR", "mean"), PSNR_std=("PSNR", "std"),
    SSIM_mean=("SSIM", "mean"), SSIM_std=("SSIM", "std"),
)
print(f"Cross-domain eval done in {time.time() - t_cd:.1f}s")

# 4. Multi-severity sweep on CBSD68 (Batched & Cached)
t_sw = time.time()
sweep_batches = {
    sev_key: (build_batched_noisy_cache(cbsd_imgs, nfn, nkw, test_seed=3000), ntype, sev_lbl)
    for sev_key, (nfn, nkw, ntype, sev_lbl) in SEVERITY_SWEEP.items()
}
sweep_rows = []
for seed in TRAIN_SEEDS:
    entries = (
        [("Literature", k, v) for k, v in lit_models_by_seed[seed].items()] +
        [("Ablation",   k, v) for k, v in ablation_models_by_seed[seed].items() if k != "Full Model (Ours)"]
    )
    for suite, mname, mobj in entries:
        for sev_key, (batches, ntype, sev_lbl) in sweep_batches.items():
            pm, sm = eval_cached_batches(mobj, batches)
            sweep_rows.append({
                "Suite": suite, "Model": mname, "Train_Seed": seed,
                "Severity_Key": sev_key, "Noise_Type": ntype, "Severity_Label": sev_lbl,
                "PSNR": pm, "SSIM": sm,
            })
            if suite == "Literature" and mname == "Noise-Adaptive (ours)":
                sweep_rows.append({
                    "Suite": "Ablation", "Model": "Full Model (Ours)", "Train_Seed": seed,
                    "Severity_Key": sev_key, "Noise_Type": ntype, "Severity_Label": sev_lbl,
                    "PSNR": pm, "SSIM": sm,
                })
del sweep_batches
df_sweep = pd.DataFrame(sweep_rows).groupby(
    ["Suite", "Model", "Severity_Key", "Noise_Type", "Severity_Label"], as_index=False
).agg(PSNR_mean=("PSNR", "mean"), PSNR_std=("PSNR", "std"), SSIM_mean=("SSIM", "mean"))
print(f"Severity sweep done in {time.time() - t_sw:.1f}s")

# 5. Out-of-Range & Unseen Noise evaluation on CBSD68 (Batched & Cached)
t_oor = time.time()
oor_batches = {
    cname: (build_batched_noisy_cache(cbsd_imgs, nfn, nkw, test_seed=4000), ctype)
    for cname, (nfn, nkw, ctype) in OOR_UNSEEN_CONDITIONS.items()
}
oor_rows = []
for seed in TRAIN_SEEDS:
    for mname, mobj in lit_models_by_seed[seed].items():
        for cname, (batches, ctype) in oor_batches.items():
            pm, sm = eval_cached_batches(mobj, batches)
            oor_rows.append({
                "Model": mname, "Train_Seed": seed, "Condition": cname, "Category": ctype,
                "PSNR": pm, "SSIM": sm,
            })
del oor_batches
if torch.cuda.is_available():
    torch.cuda.empty_cache()
df_oor = pd.DataFrame(oor_rows).groupby(["Model", "Condition", "Category"], as_index=False).agg(
    PSNR_mean=("PSNR", "mean"), PSNR_std=("PSNR", "std"),
    SSIM_mean=("SSIM", "mean"), SSIM_std=("SSIM", "std"),
)
print(f"Out-of-range & unseen noise eval done in {time.time() - t_oor:.1f}s")

df_complexity.to_csv(os.path.join(CSV_DIR, "combined_complexity.csv"), index=False)
df_multids_seeds.to_csv(os.path.join(CSV_DIR, "combined_multidataset_per_seed.csv"), index=False)
df_multids.to_csv(os.path.join(CSV_DIR, "combined_multidataset.csv"), index=False)
df_cross_domain.to_csv(os.path.join(CSV_DIR, "cross_domain_results.csv"), index=False)
df_sweep.to_csv(os.path.join(CSV_DIR, "combined_multiseverity.csv"), index=False)
df_oor.to_csv(os.path.join(CSV_DIR, "out_of_range_unseen_noise.csv"), index=False)
print("Saved all multi-seed CSVs.")

## 5. Seed-Aware Statistical Tests, Rankings, Excel Workbook & Clean IEEE LaTeX Tables

- **Seed-level paired *t*-tests** across the 3 training seeds (honest variance accounting for training randomness).
- **Raw-string LaTeX generation** (`r"\begin{table*}..."`, `r"\toprule"`, `r"\bottomrule"`) verified to contain zero control characters.

In [ ]:
# 1. Noise Descriptor Head Calibration (averaged over 3 seeds)
def eval_noise_head_calib_3seed(imgs, n_samples=150, seed=2026):
    rng = np.random.default_rng(seed)
    samples = []
    for i in range(n_samples):
        img = imgs[i % len(imgs)]
        nt = int(rng.integers(0, 3))
        sigma, density = float(rng.uniform(15, 50)), float(rng.uniform(0.05, 0.30))
        if nt == 0:
            noisy = add_gaussian_noise(img, sigma); gt_g, gt_i = sigma / 50.0, 0.0
        elif nt == 1:
            noisy = add_impulse_noise(img, density); gt_g, gt_i = 0.0, density / 0.30
        else:
            noisy = add_mixed_noise(img, sigma * 0.6, density * 0.6)
            gt_g, gt_i = (sigma * 0.6) / 50.0, (density * 0.6) / 0.30
        samples.append((nt, gt_g, gt_i, torch.from_numpy(noisy / 255.0).permute(2, 0, 1).unsqueeze(0).float().to(device)))

    calib_rows, scatter_rows = [], []
    for vname, cfg in ABLATION_CONFIGS.items():
        if not cfg["kwargs"]["use_film"]:
            continue
        maes_g, maes_i, r2s_g, r2s_i = [], [], [], []
        for s_idx, tseed in enumerate(TRAIN_SEEDS):
            m = ablation_models_by_seed[tseed][vname]
            m.eval()
            pg_l, pi_l, gg_l, gi_l = [], [], [], []
            with torch.no_grad():
                for nt, gt_g, gt_i, noisy_t in samples:
                    _, _, desc = m(noisy_t)
                    pg, pi = float(desc[0, 0].item()), float(desc[0, 1].item())
                    pg_l.append(pg); pi_l.append(pi); gg_l.append(gt_g); gi_l.append(gt_i)
                    if s_idx == 0 and vname == "Full Model (Ours)":
                        scatter_rows.append({
                            "Noise_Type": ["Gaussian", "Impulse", "Mixed"][nt],
                            "GT_Gaussian": gt_g, "Pred_Gaussian": pg,
                            "GT_Impulse": gt_i, "Pred_Impulse": pi,
                        })
            pg_a, pi_a, gg_a, gi_a = map(np.array, (pg_l, pi_l, gg_l, gi_l))
            maes_g.append(np.mean(np.abs(pg_a - gg_a)))
            maes_i.append(np.mean(np.abs(pi_a - gi_a)))
            r2s_g.append(stats.pearsonr(gg_a, pg_a)[0] ** 2 if np.std(pg_a) > 1e-8 else 0.0)
            r2s_i.append(stats.pearsonr(gi_a, pi_a)[0] ** 2 if np.std(pi_a) > 1e-8 else 0.0)
        calib_rows.append({
            "Variant": vname,
            "MAE_Gaussian_mean": round(float(np.mean(maes_g)), 4),
            "R2_Gaussian_mean":  round(float(np.mean(r2s_g)), 4),
            "MAE_Impulse_mean":  round(float(np.mean(maes_i)), 4),
            "R2_Impulse_mean":   round(float(np.mean(r2s_i)), 4),
        })
    return pd.DataFrame(calib_rows), pd.DataFrame(scatter_rows)


df_calib, df_scatter = eval_noise_head_calib_3seed(cbsd_imgs)
df_calib.to_csv(os.path.join(CSV_DIR, "noise_head_accuracy.csv"), index=False)

# 2. Seed-level Paired t-test on CBSD68 (across the 3 training seeds)
stat_rows = []
for suite, ref_name in [("Literature", "Noise-Adaptive (ours)"), ("Ablation", "Full Model (Ours)")]:
    for cond_name in test_conditions.keys():
        ref_vals = np.array(per_seed_cbsd_means[(suite, ref_name, cond_name)])
        for (s2, mname, c2), alt_vals in per_seed_cbsd_means.items():
            if s2 != suite or c2 != cond_name or mname == ref_name:
                continue
            alt_arr = np.array(alt_vals)
            diff = ref_vals - alt_arr
            t_stat, t_p = stats.ttest_rel(ref_vals, alt_arr)
            stat_rows.append({
                "Suite": suite, "Reference": ref_name, "Comparison_Model": mname,
                "Condition": cond_name,
                "Delta_PSNR_mean_dB": round(float(diff.mean()), 4),
                "Delta_PSNR_std_dB":  round(float(diff.std(ddof=1)), 4),
                "Seed_Paired_t_p":    float(t_p),
            })
df_stats = pd.DataFrame(stat_rows)
df_stats.to_csv(os.path.join(CSV_DIR, "statistical_tests_3seed.csv"), index=False)


# 3. Suite Summary & Ranking Tables
def build_suite_summary(suite_name, ref_name):
    sub = df_multids[df_multids["Suite"] == suite_name].copy()
    psnr_w = sub.pivot(index="Model", columns="Cell", values="PSNR_mean")
    rank_w = psnr_w.rank(ascending=False, method="min")
    rank_w["Mean_Rank"] = rank_w.mean(axis=1)
    rank_w["Overall_Rank"] = rank_w["Mean_Rank"].rank(ascending=True, method="min").astype(int)

    agg = sub.groupby("Model").agg(Mean_PSNR=("PSNR_mean", "mean"), Mean_SSIM=("SSIM_mean", "mean")).reset_index()
    cbsd = sub[sub["Dataset"] == "CBSD68"].pivot(index="Model", columns="Condition", values="PSNR_mean").rename(
        columns={"Gaussian": "CBSD68_Gaussian_PSNR", "Impulse": "CBSD68_Impulse_PSNR", "Mixed": "CBSD68_Mixed_PSNR"}
    )
    comp_sub = df_complexity[df_complexity["Suite"] == suite_name].drop(columns=["Suite"])
    out = agg.merge(cbsd, on="Model").merge(comp_sub, on="Model", how="left").merge(
        rank_w[["Mean_Rank", "Overall_Rank"]].reset_index(), on="Model"
    )
    ref_val = float(out.loc[out["Model"] == ref_name, "Mean_PSNR"].iloc[0])
    out["Delta_PSNR_vs_Ours"] = out["Mean_PSNR"] - ref_val
    out = out.sort_values("Overall_Rank").reset_index(drop=True)
    return out, psnr_w, rank_w


df_lit_summary, lit_psnr_w, lit_rank_w = build_suite_summary("Literature", "Noise-Adaptive (ours)")
df_abl_summary, abl_psnr_w, abl_rank_w = build_suite_summary("Ablation", "Full Model (Ours)")

df_lit_summary.to_csv(os.path.join(CSV_DIR, "literature_comparison_summary.csv"), index=False)
df_abl_summary.to_csv(os.path.join(CSV_DIR, "ablation_summary.csv"), index=False)

print("=== LITERATURE COMPARISON (3-SEED MEAN) ===")
print(df_lit_summary[["Overall_Rank", "Model", "CBSD68_Gaussian_PSNR", "CBSD68_Impulse_PSNR",
                      "CBSD68_Mixed_PSNR", "Mean_PSNR", "Mean_SSIM", "Mean_Rank", "Params_K", "GMac_256"]].round(3).to_string(index=False))

print("\n=== ABLATION STUDY (3-SEED MEAN) ===")
print(df_abl_summary[["Overall_Rank", "Model", "CBSD68_Gaussian_PSNR", "CBSD68_Impulse_PSNR",
                      "CBSD68_Mixed_PSNR", "Mean_PSNR", "Delta_PSNR_vs_Ours", "Mean_SSIM", "Mean_Rank"]].round(3).to_string(index=False))

# 4. Multi-Sheet Excel Export
xlsx_path = os.path.join(RESULTS_DIR, "ieee_ablation_and_comparison_results.xlsx")
with pd.ExcelWriter(xlsx_path, engine="openpyxl") as writer:
    df_lit_summary.round(4).to_excel(writer, sheet_name="Literature_Comparison", index=False)
    df_abl_summary.round(4).to_excel(writer, sheet_name="Ablation_Summary", index=False)
    df_cross_domain.round(4).to_excel(writer, sheet_name="Cross_Domain_Eval", index=False)
    df_oor.round(4).to_excel(writer, sheet_name="OutOfRange_UnseenNoise", index=False)
    df_multids.round(4).to_excel(writer, sheet_name="Multi_Dataset_Eval", index=False)
    df_sweep.round(4).to_excel(writer, sheet_name="Multi_Severity_Eval", index=False)
    lit_rank_w.reset_index().round(3).to_excel(writer, sheet_name="Literature_Rankings", index=False)
    abl_rank_w.reset_index().round(3).to_excel(writer, sheet_name="Ablation_Rankings", index=False)
    df_calib.round(4).to_excel(writer, sheet_name="Noise_Head_Accuracy", index=False)
    df_stats.to_excel(writer, sheet_name="Statistical_Tests", index=False)
    for _, ws in writer.sheets.items():
        ws.freeze_panes = "A2"
        for col in ws.columns:
            max_len = max(len(str(c.value or "")) for c in col)
            ws.column_dimensions[col[0].column_letter].width = min(max_len + 4, 34)
print(f"\nSaved Excel workbook: {xlsx_path}")


# 5. Clean Raw-String IEEE LaTeX Tables (Zero control characters)
def write_ieee_table(df_sum, suite_name, caption, label, tex_name):
    cbsd = df_multids[(df_multids["Suite"] == suite_name) & (df_multids["Dataset"] == "CBSD68")].set_index(["Model", "Condition"])
    mcols = ["CBSD68_Gaussian_PSNR", "CBSD68_Impulse_PSNR", "CBSD68_Mixed_PSNR", "Mean_PSNR", "Mean_SSIM"]
    top1 = {c: df_sum[c].nlargest(1).iloc[0] for c in mcols}
    top2 = {c: df_sum[c].nlargest(2).iloc[1] for c in mcols}

    def fmt(val, col, std=None):
        s = f"{val:.2f}" if "PSNR" in col else f"{val:.4f}"
        if std is not None:
            s = rf"{s}$\pm${std:.2f}"
        if abs(val - top1[col]) < 1e-6:
            return r"\textbf{" + s + "}"
        if abs(val - top2[col]) < 1e-6:
            return r"\underline{" + s + "}"
        return s

    lines = [
        r"\begin{table*}[t]",
        r"\centering",
        r"\caption{" + caption + "}",
        r"\label{" + label + "}",
        r"\setlength{\tabcolsep}{5pt}",
        r"\begin{tabular}{lccccccc}",
        r"\toprule",
        r"Model & Params (K) & Gaussian ($\sigma=25$) & Impulse ($p=0.15$) & Mixed & Mean PSNR & Mean SSIM & Avg. Rank \\",
        r"\midrule",
    ]
    for _, row in df_sum.iterrows():
        m = row["Model"]
        pk = f"{row['Params_K']:.1f}" if pd.notna(row["Params_K"]) else "--"
        gs = fmt(row["CBSD68_Gaussian_PSNR"], "CBSD68_Gaussian_PSNR", cbsd.loc[(m, "Gaussian"), "PSNR_std"])
        is_ = fmt(row["CBSD68_Impulse_PSNR"], "CBSD68_Impulse_PSNR", cbsd.loc[(m, "Impulse"), "PSNR_std"])
        ms = fmt(row["CBSD68_Mixed_PSNR"], "CBSD68_Mixed_PSNR", cbsd.loc[(m, "Mixed"), "PSNR_std"])
        mp = fmt(row["Mean_PSNR"], "Mean_PSNR")
        mss = fmt(row["Mean_SSIM"], "Mean_SSIM")
        rk = rf"{row['Mean_Rank']:.2f} (\# {int(row['Overall_Rank'])})".replace("# ", "#")
        m_tex = m.replace("_", r"\_")
        lines.append(rf"{m_tex} & {pk} & {gs} & {is_} & {ms} & {mp} & {mss} & {rk} \\")
    lines.extend([r"\bottomrule", r"\end{tabular}", r"\end{table*}"])

    tex_out = "\n".join(lines)
    assert "\x08" not in tex_out and "\t" not in tex_out, "Control character detected in LaTeX!"
    out_p = os.path.join(RESULTS_DIR, tex_name)
    with open(out_p, "w") as f:
        f.write(tex_out)
    print(f"Saved clean LaTeX table: {out_p}")


write_ieee_table(
    df_lit_summary, "Literature",
    r"Quantitative Comparison Against Literature-Style Baselines on Held-Out CBSD68 and Cross-Dataset Benchmarks (Mean $\pm$ Std over 3 Training Seeds). Best in \textbf{bold}, second-best \underline{underlined}.",
    "tab:literature_comparison", "ieee_comparison_table.tex"
)
write_ieee_table(
    df_abl_summary, "Ablation",
    r"Ablation Study of the Proposed Noise-Adaptive Architecture (Mean $\pm$ Std over 3 Training Seeds). Best in \textbf{bold}, second-best \underline{underlined}.",
    "tab:ablation_study", "ieee_ablation_table.tex"
)

## 6. IEEE Publication Figures & Colorblind-Safe Heatmaps

Generates 300-DPI PNG + vector PDF figures for every section of the IEEE Results:
1. **Fig. 1**: Literature Comparison PSNR & Rank Heatmap (incl. 44K Matched `HKCNN-style`)
2. **Fig. 2**: Ablation Study PSNR & Rank Heatmap
3. **Fig. 3**: Cross-Domain Generalization Heatmap (Single-Noise Specialists vs. `Noise-Adaptive (ours)`)
4. **Fig. 4**: Out-of-Range & Unseen Noise Generalization Heatmap
5. **Fig. 5**: Efficiency–Accuracy Pareto Frontier (Params & GMac vs. Mean PSNR)
6. **Fig. 6**: Multi-Severity Curves & Noise-Head Calibration

In [ ]:
plt.rcParams.update({
    "font.family": "serif",
    "font.serif": ["Times New Roman", "DejaVu Serif", "Liberation Serif"],
    "font.size": 9, "axes.titlesize": 10, "axes.titleweight": "bold",
    "axes.labelsize": 9, "xtick.labelsize": 8, "ytick.labelsize": 8,
    "legend.fontsize": 7.5, "figure.dpi": 150, "savefig.dpi": 300,
    "axes.spines.top": False, "axes.spines.right": False,
})

IEEE_PSNR_CMAP = sns.color_palette("YlGnBu", as_cmap=True)
IEEE_RANK_CMAP = sns.color_palette("Blues_r", as_cmap=True)


def save_ieee_fig(fig, stem):
    fig.savefig(os.path.join(FIGURES_DIR, f"{stem}.png"), dpi=300, bbox_inches="tight")
    fig.savefig(os.path.join(FIGURES_DIR, f"{stem}.pdf"), bbox_inches="tight")
    print(f"Saved {stem}.png & .pdf")


ordered_cells = [
    "CBSD68_Gaussian", "CBSD68_Impulse", "CBSD68_Mixed",
    "Set12_Gaussian",  "Set12_Impulse",  "Set12_Mixed",
    "Kodak24_Gaussian","Kodak24_Impulse","Kodak24_Mixed",
]
pretty_cols = [
    "CBSD68\nGauss.", "CBSD68\nImp.", "CBSD68\nMixed",
    "Set12\nGauss.",  "Set12\nImp.",  "Set12\nMixed",
    "Kodak24\nGauss.","Kodak24\nImp.","Kodak24\nMixed",
]


def plot_psnr_rank_heatmap(df_sum, psnr_w, rank_w, title_prefix, stem):
    order = df_sum["Model"].tolist()
    p_mat = psnr_w.loc[order, ordered_cells]
    r_mat = rank_w.loc[order, ordered_cells]
    z_mat = (p_mat - p_mat.mean(axis=0)) / (p_mat.std(axis=0) + 1e-8)

    annot = np.empty(p_mat.shape, dtype=object)
    for r in range(p_mat.shape[0]):
        for c in range(p_mat.shape[1]):
            annot[r, c] = f"{p_mat.iloc[r, c]:.2f}\n(#{int(r_mat.iloc[r, c])})"

    fig, (ax1, ax2) = plt.subplots(
        1, 2, figsize=(10.5, 0.52 * len(order) + 1.2),
        gridspec_kw={"width_ratios": [4.2, 1.1], "wspace": 0.08}
    )
    sns.heatmap(
        z_mat.values, ax=ax1, cmap=IEEE_PSNR_CMAP, annot=annot, fmt="",
        annot_kws={"size": 7.5}, linewidths=0.7, linecolor="white",
        xticklabels=pretty_cols, yticklabels=order,
        cbar_kws={"label": "Column-Normalized PSNR (z-score)", "shrink": 0.85},
    )
    ax1.set_title(f"(a) {title_prefix}: Cross-Dataset PSNR (dB) & Rank Heatmap")
    ax1.tick_params(axis="x", rotation=0)

    mr = df_sum.set_index("Model").loc[order, "Mean_Rank"]
    cols = ["#08519c" if i == 0 else ("#3182bd" if i == 1 else "#9ecae1") for i in range(len(order))]
    y_pos = np.arange(len(order)) + 0.5
    ax2.barh(y_pos, mr.values, height=0.65, color=cols, edgecolor="#252525", linewidth=0.5)
    ax2.set_ylim(len(order), 0); ax2.set_yticks([])
    ax2.set_xlabel("Mean Rank (Lower = Better)"); ax2.set_title("(b) Avg. Rank")
    ax2.set_xlim(0, max(mr.values) * 1.28)
    for y, rk in zip(y_pos, mr.values):
        ax2.text(rk + 0.1, y, f"{rk:.2f}", va="center", fontsize=8, fontweight="bold")
    save_ieee_fig(fig, stem)
    plt.show()


plot_psnr_rank_heatmap(df_lit_summary, lit_psnr_w, lit_rank_w, "Literature Comparison (3-Seed)", "fig1_literature_psnr_rank_heatmap")
plot_psnr_rank_heatmap(df_abl_summary, abl_psnr_w, abl_rank_w, "Ablation Study (3-Seed)", "fig2_ablation_psnr_rank_heatmap")

# Fig 3: Cross-Domain Generalization Heatmap
cd_piv = df_cross_domain.pivot(index="Model", columns="Condition", values="PSNR_mean")[["Gaussian", "Impulse", "Mixed"]]
cd_order = ["Gaussian-only (LWIFCNN)", "Impulse-only (LWIFCNN)", "Mixed-trained (LWIFCNN)", "Noise-Adaptive (ours)"]
cd_piv = cd_piv.loc[cd_order]
fig, ax = plt.subplots(figsize=(6.2, 3.2))
sns.heatmap(cd_piv, ax=ax, cmap=IEEE_PSNR_CMAP, annot=True, fmt=".2f", linewidths=0.7, linecolor="white",
            cbar_kws={"label": "CBSD68 PSNR (dB)"})
ax.set_title("Cross-Domain Generalization: Specialists vs. Noise-Adaptive (3-Seed Mean)")
save_ieee_fig(fig, "fig3_cross_domain_heatmap")
plt.show()

# Fig 4: Out-of-Range & Unseen Noise Heatmap
oor_piv = df_oor.pivot(index="Model", columns="Condition", values="PSNR_mean")
oor_z = (oor_piv - oor_piv.mean(axis=0)) / (oor_piv.std(axis=0) + 1e-8)
fig, ax = plt.subplots(figsize=(9.5, 4.0))
sns.heatmap(oor_z, ax=ax, cmap=IEEE_PSNR_CMAP, annot=oor_piv.values, fmt=".2f", linewidths=0.7, linecolor="white",
            cbar_kws={"label": "Column-Normalized PSNR (z-score)"})
ax.set_title("Out-of-Range Severities & Unseen Noise Types (CBSD68, 3-Seed Mean)")
ax.tick_params(axis="x", rotation=20)
save_ieee_fig(fig, "fig4_out_of_range_unseen_noise")
plt.show()

# Fig 5: Efficiency-Accuracy Pareto Frontier
df_all_sum = pd.concat([
    df_lit_summary.assign(Suite="Literature"),
    df_abl_summary[df_abl_summary["Model"] != "Full Model (Ours)"].assign(Suite="Ablation"),
], ignore_index=True).dropna(subset=["Params_K", "GMac_256"])

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(9.4, 3.8))
for _, row in df_all_sum.iterrows():
    is_ours = row["Model"] == "Noise-Adaptive (ours)"
    c = "#cb181d" if is_ours else ("#08519c" if row["Suite"] == "Literature" else "#238b45")
    m = "*" if is_ours else ("o" if row["Suite"] == "Literature" else "s")
    sz = 170 if is_ours else 60
    ax1.scatter(row["Params_K"], row["Mean_PSNR"], color=c, marker=m, s=sz, zorder=5, edgecolor="black", linewidth=0.5)
    ax1.annotate(row["Model"], (row["Params_K"], row["Mean_PSNR"]), xytext=(4, 3), textcoords="offset points", fontsize=6.5)
    ax2.scatter(row["GMac_256"], row["Mean_PSNR"], color=c, marker=m, s=sz, zorder=5, edgecolor="black", linewidth=0.5)
    ax2.annotate(row["Model"], (row["GMac_256"], row["Mean_PSNR"]), xytext=(4, 3), textcoords="offset points", fontsize=6.5)
ax1.set_xlabel("Trainable Parameters (K)"); ax1.set_ylabel("Mean PSNR (dB)"); ax1.set_title("(a) Parameters vs. Mean PSNR"); ax1.grid(True, linestyle=":", alpha=0.5)
ax2.set_xlabel("Compute (GMac @ 256$\times$256)"); ax2.set_ylabel("Mean PSNR (dB)"); ax2.set_title("(b) Compute vs. Mean PSNR"); ax2.grid(True, linestyle=":", alpha=0.5)
fig.tight_layout()
save_ieee_fig(fig, "fig5_efficiency_pareto")
plt.show()

# Fig 6: Noise Descriptor Calibration
if len(df_scatter) > 0:
    fig, axes = plt.subplots(1, 2, figsize=(8.8, 3.5))
    for nt, col in {"Gaussian": "#08519c", "Impulse": "#d95f02", "Mixed": "#238b45"}.items():
        s = df_scatter[df_scatter["Noise_Type"] == nt]
        axes[0].scatter(s["GT_Gaussian"], s["Pred_Gaussian"], color=col, alpha=0.75, s=26, label=nt, edgecolor="white", linewidth=0.4)
        axes[1].scatter(s["GT_Impulse"],  s["Pred_Impulse"],  color=col, alpha=0.75, s=26, label=nt, edgecolor="white", linewidth=0.4)
    for ax, t, xl, yl in [
        (axes[0], "(a) Gaussian Strength $\hat{d}_{\mathrm{gauss}}$", "GT Normalized $\sigma$", "Predicted $\hat{d}_{\mathrm{gauss}}$"),
        (axes[1], "(b) Impulse Density $\hat{d}_{\mathrm{imp}}$", "GT Normalized Density", "Predicted $\hat{d}_{\mathrm{imp}}$"),
    ]:
        ax.plot([0, 1], [0, 1], "k--", linewidth=1.0)
        ax.set_xlim(-0.03, 1.03); ax.set_ylim(-0.03, 1.03)
        ax.set_title(t); ax.set_xlabel(xl); ax.set_ylabel(yl); ax.grid(True, linestyle=":", alpha=0.5); ax.legend(loc="upper left")
    fig.tight_layout()
    save_ieee_fig(fig, "fig6_noise_head_calibration")
    plt.show()